In [2]:
import pandas as pd

df = pd.read_csv("initial_projects.csv")

# Replace "column_name" with the column you want to deduplicate on
df_unique = df.drop_duplicates(subset="repo_name")

df_unique.to_csv("output.csv", index=False)

In [3]:
import pandas as pd

df = pd.read_csv("initial_projects.csv")

col = "repo_name"   # ← change to your column

# Create a lowercase version for comparison
df["_lower"] = df[col].str.lower()

# Drop duplicates based on the lowercase version
df_unique = df.drop_duplicates(subset="_lower")

# Remove helper column
df_unique = df_unique.drop(columns=["_lower"])

df_unique.to_csv("output.csv", index=False)

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("selected_projects_add.csv")

# ------------------------------------------------------------------
# 0) Basic cleanup & helper columns
# ------------------------------------------------------------------

# Filter out archived or obvious junk if you want
df = df[df["is_archived"] != True]
df = df[df["language"] == "Java"]

# Avoid division by zero
df["src_files"] = df["src_files"].replace(0, np.nan)
df["total_pull_requests"] = df["total_pull_requests"].fillna(0)
df["total_issues"] = df["total_issues"].fillna(0)

# Test ratio: how test-heavy they are (relative to code size)
df["test_ratio"] = df["test_files"] / df["src_files"]

# Closed issue ratio
df["closed_issue_ratio"] = df["closed_issues"] / df["total_issues"]
df["closed_issue_ratio"] = df["closed_issue_ratio"].fillna(0)

# PR merged ratio
df["merged_pr_ratio"] = df["merged_pull_requests"] / df["total_pull_requests"]
df["merged_pr_ratio"] = df["merged_pr_ratio"].fillna(0)

# Recency: days since last push (smaller = better)
df["pushed_at"] = pd.to_datetime(df["pushed_at"], errors="coerce", utc=True)

max_date = pd.Timestamp.now(tz="UTC")

df["days_since_push"] = (max_date - df["pushed_at"]).dt.days
df["days_since_push"] = df["days_since_push"].fillna(df["days_since_push"].max())


# Small log transforms to compress huge ranges
df["log_stars"] = np.log1p(df["stars"].fillna(0))
df["log_forks"] = np.log1p(df["forks"].fillna(0))
df["log_total_prs"] = np.log1p(df["total_pull_requests"])
df["log_test_files"] = np.log1p(df["test_files"].fillna(0))
df["log_commit_count"] = np.log1p(df["commit_count"].fillna(0))
df["log_releases"] = np.log1p(df["releases_count"].fillna(0))
df["log_mentionable_users"] = np.log1p(df["mentionable_users"].fillna(0))

# Make sure CI/test flags are numeric (0/1) not NaN
for col in ["has_actions", "workflow_runs_tests"]:
    if col in df.columns:
        df[col] = df[col].fillna(False).astype(int)

# ------------------------------------------------------------------
# 1) Build component scores (all between 0 and 1 via rank(pct=True))
# ------------------------------------------------------------------

def pct_rank(series):
    return series.rank(pct=True).fillna(0)

# Popularity: stars + forks (watchers usually correlated with stars)
df["popularity_score"] = (
    0.7 * pct_rank(df["log_stars"]) +
    0.3 * pct_rank(df["log_forks"])
)

# Activity / maintenance:
#   - more recent pushes (invert days_since_push)
#   - more commits
#   - more releases
#   - higher closed_issue_ratio
df["activity_score"] = (
    0.35 * (1 - pct_rank(df["days_since_push"])) +   # recent pushes
    0.25 * pct_rank(df["log_commit_count"]) +        # big history
    0.20 * pct_rank(df["log_releases"]) +            # releases
    0.20 * pct_rank(df["closed_issue_ratio"])        # issues handled
)

# PR culture:
#   - many PRs
#   - high merged ratio
df["pr_score"] = (
    0.6 * pct_rank(df["log_total_prs"]) +
    0.4 * pct_rank(df["merged_pr_ratio"])
)

# Testing intensity:
#   - test_ratio (tests per src file)
#   - absolute # test files
#   - CI & workflows that seem to run tests
df["testing_score"] = (
    0.45 * pct_rank(df["test_ratio"]) +
    0.25 * pct_rank(df["log_test_files"]) +
    0.15 * df["has_actions"] +
    0.15 * df["workflow_runs_tests"]
)

# Community:
#   - many mentionable users (contributors)
df["community_score"] = pct_rank(df["log_mentionable_users"])

# ------------------------------------------------------------------
# 2) Overall score (weights you can tweak)
# ------------------------------------------------------------------
df["overall_score"] = (
    0.30 * df["popularity_score"] +
    0.25 * df["testing_score"] +
    0.20 * df["activity_score"] +
    0.15 * df["pr_score"] +
    0.10 * df["community_score"]
)

# ------------------------------------------------------------------
# 3) Inspect and pick top 200 projects
# ------------------------------------------------------------------
# Filter out projects that do not care about testing
df_filtered = df[df["test_ratio"] >= 0.01]

# Sort again AFTER filtering
df_sorted = df_filtered.sort_values("overall_score", ascending=False)
df_sorted

,Name,local_build,sat_usage,ci_build_combined,ci_state,ci_usage,cr_reviews,cr_comments,cr_changed_files,cr_changed_lines,...,log_test_files,log_commit_count,log_releases,log_mentionable_users,popularity_score,activity_score,pr_score,testing_score,community_score,overall_score
1064,quarkusio/quarkus,NaN,0.0,0.0,0.0,1.0,56.0,38.0,101.0,3065.0,...,9.364862,10.927915,6.075346,7.037028,0.934870,0.861078,0.891896,0.981596,0.972119,0.918561
142,apache/shardingsphere,1.0,1.0,1.0,1.0,1.0,49.0,21.0,189.0,6140.0,...,8.095599,10.765216,4.077537,6.452049,0.960967,0.772212,0.979480,0.909236,0.910037,0.904628
417,elastic/elasticsearch,0.0,1.0,NaN,NaN,0.0,37.0,42.0,103.0,252289.0,...,9.107532,11.427977,5.398163,8.267192,0.997100,0.866729,0.908401,0.706243,0.996283,0.903057
735,keycloak/keycloak,0.0,0.0,1.0,0.0,1.0,63.0,33.0,410.0,3057.0,...,7.702104,10.297049,4.553877,7.252762,0.978810,0.809257,0.851450,0.802728,0.979926,0.884649
1224,spring-projects/spring-boot,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,...,8.296048,10.981115,5.802118,7.085064,0.998513,0.927732,0.560595,0.951728,0.973234,0.883689
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746,kiegroup/optaplanner-quickstarts,NaN,0.0,NaN,NaN,0.0,27.0,90.0,421.0,1732.0,...,3.891820,6.061457,0.000000,4.189655,0.015539,0.072156,0.030781,0.567892,0.354275,0.149468
1497,neuroph/neuroph,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.007333,6.729824,0.000000,3.135494,0.229071,0.074870,0.163569,0.143514,0.057249,0.147404
1499,jensstein/oandbackup,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.098612,6.687109,0.000000,2.397895,0.331524,0.103978,0.050260,0.073917,0.007807,0.147372
1549,GateNLP/gate-core,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.583519,6.095825,2.079442,3.401197,0.008810,0.185130,0.055167,0.438644,0.102230,0.135979


In [17]:
import numpy as np
import pandas as pd

# ---------- 1. Load data ----------
df = pd.read_csv("./selected_projects_java.csv")

# Optional: drop archived repos
df = df[df["is_archived"] != True]
df = df[df["language"] == "Java"]

# ---------- 2. Helper + preprocessing ----------

def pct_rank(series):
    """Percentile rank in [0,1]."""
    return series.rank(pct=True).fillna(0.0)

def normalize_java_version(val):
    if pd.isna(val):
        return float("nan")
    val = str(val).strip()
    # convert "1.8" → "8"
    if val.startswith("1."):
        try:
            return int(val.split(".")[1])
        except:
            return float("nan")
    # normal integer versions "8", "11", "17", "21"
    try:
        return int(val)
    except:
        return float("nan")

# Avoid div-by-zero / NaNs
df["src_files"] = df["src_files"].replace(0, np.nan)
df["total_pull_requests"] = df["total_pull_requests"].fillna(0)
df["total_issues"] = df["total_issues"].fillna(0)
df["merged_pull_requests"] = df["merged_pull_requests"].fillna(0)
df["closed_pull_requests"] = df["closed_pull_requests"].fillna(0)
df["closed_issues"] = df["closed_issues"].fillna(0)
df["mentionable_users"] = df["mentionable_users"].fillna(0)
df["unique_pr_authors"] = df["unique_pr_authors"].fillna(0)
df["releases_count"] = df["releases_count"].fillna(0)
df["commit_count"] = df["commit_count"].fillna(0)
df["test_files"] = df["test_files"].fillna(0)
df["stars"] = df["stars"].fillna(0)
df["forks"] = df["forks"].fillna(0)

# Test ratio
df["test_ratio"] = df["test_files"] / df["src_files"]

# Closed issue ratio
df["closed_issue_ratio"] = df["closed_issues"] / df["total_issues"]
df["closed_issue_ratio"] = df["closed_issue_ratio"].fillna(0)

# PR merged ratio
df["merged_pr_ratio"] = df["merged_pull_requests"] / df["total_pull_requests"]
df["merged_pr_ratio"] = df["merged_pr_ratio"].fillna(0)

# ⬅️ NEW: Closed PR ratio
df["closed_pr_ratio"] = df["closed_pull_requests"] / df["total_pull_requests"]
df["closed_pr_ratio"] = df["closed_pr_ratio"].fillna(0)

# Unique PR authors log scale
df["log_unique_pr_authors"] = np.log1p(df["unique_pr_authors"])

# Normalize timestamps to UTC
df["pushed_at"] = pd.to_datetime(df["pushed_at"], errors="coerce", utc=True)
now_utc = pd.Timestamp.now(tz="UTC")
df["days_since_push"] = (now_utc - df["pushed_at"]).dt.days
df["days_since_push"] = df["days_since_push"].fillna(df["days_since_push"].max())

# Log transforms
df["log_stars"] = np.log1p(df["stars"])
df["log_forks"] = np.log1p(df["forks"])
df["log_total_prs"] = np.log1p(df["total_pull_requests"])
df["log_test_files"] = np.log1p(df["test_files"])
df["log_commit_count"] = np.log1p(df["commit_count"])
df["log_releases"] = np.log1p(df["releases_count"])
df["log_mentionable_users"] = np.log1p(df["mentionable_users"])

# Ensure flags are numeric 0/1
for col in ["has_actions", "workflow_runs_tests", "has_contributing"]:
    if col in df.columns:
        df[col] = df[col].fillna(False).astype(int)
    else:
        df[col] = 0

# ---------- 3. Component scores ----------

# 3.1 Popularity
df["popularity_score"] = (
    0.7 * pct_rank(df["log_stars"]) +
    0.3 * pct_rank(df["log_forks"])
)

# 3.2 Activity / Maintenance
df["activity_score"] = (
    0.35 * (1.0 - pct_rank(df["days_since_push"])) +
    0.25 * pct_rank(df["log_commit_count"]) +
    0.20 * pct_rank(df["log_releases"]) +
    0.20 * pct_rank(df["closed_issue_ratio"])
)

# 3.3 PR culture (UPDATED)
df["pr_score"] = (
    0.45 * pct_rank(df["log_total_prs"]) +
    0.30 * pct_rank(df["merged_pr_ratio"]) +
    0.25 * pct_rank(df["closed_pr_ratio"])     # ⬅️ NEW
)

# 3.4 Testing intensity
df["testing_score"] = (
    0.45 * pct_rank(df["test_ratio"]) +
    0.25 * pct_rank(df["log_test_files"]) +
    0.15 * df["has_actions"] +
    0.15 * df["workflow_runs_tests"]
)

# 3.5 Community
df["community_score"] = (
    0.6 * pct_rank(df["log_mentionable_users"]) +
    0.4 * pct_rank(df["log_unique_pr_authors"])
)

# 3.6 Maturity
df["maturity_score"] = (
    0.5 * df["has_contributing"] +
    0.25 * df["has_actions"] +
    0.25 * df["workflow_runs_tests"]
)

# ---------- 4. Final weighted score ----------
df["overall_score"] = (
    0.30 * df["popularity_score"] +
    0.25 * df["testing_score"] +
    0.20 * df["activity_score"] +
    0.15 * df["pr_score"] +
    0.10 * df["community_score"] +
    0.05 * df["maturity_score"]
)

df_filtered = df[df["test_ratio"] >= 0.01]
df_filtered = df_filtered[df_filtered["test_files"] >= 10]
df_filtered["java_version_num"] = df_filtered["java_version"].apply(normalize_java_version)

# Keep rows where:
#   - java_version_num <= 21
#   - OR java_version_num is NaN
# df_filtered = df_filtered[
#     (df_filtered["java_version_num"].isna()) |
#     (df_filtered["java_version_num"] <= 21)
# ]
df_filtered = df_filtered[df_filtered["java_version_num"] <= 21]

# Sort again AFTER filtering
df_sorted = df_filtered.sort_values("overall_score", ascending=False)
df_sorted

,Name,local_build,sat_usage,ci_build_combined,ci_state,ci_usage,cr_reviews,cr_comments,cr_changed_files,cr_changed_lines,...,log_releases,log_mentionable_users,popularity_score,activity_score,pr_score,testing_score,community_score,maturity_score,overall_score,java_version_num
1234,spring-projects/spring-framework,0.0,1.0,0.0,0.0,1.0,0.0,24.0,23.0,108.0,...,5.863631,6.911747,0.996205,0.918806,0.657924,0.914458,0.980283,1.0,0.957954,17.0
1064,quarkusio/quarkus,NaN,0.0,0.0,0.0,1.0,56.0,38.0,101.0,3065.0,...,6.075346,7.037028,0.934821,0.855115,0.741220,0.981582,0.979092,1.0,0.955957,21.0
1224,spring-projects/spring-boot,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,...,5.802118,7.085064,0.998512,0.921819,0.670052,0.951691,0.982143,0.5,0.945562,17.0
268,checkstyle/checkstyle,1.0,1.0,0.0,0.0,1.0,8.0,22.0,234.0,9438.0,...,5.170484,6.192362,0.912723,0.804557,0.712760,0.981383,0.920089,1.0,0.928997,17.0
142,apache/shardingsphere,1.0,1.0,1.0,1.0,1.0,49.0,21.0,189.0,6140.0,...,4.077537,6.452049,0.960938,0.766053,0.747061,0.909167,0.931994,1.0,0.924042,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1479,Hive2Hive/Hive2Hive,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.197225,2.484907,0.264993,0.210565,0.046001,0.326358,0.009077,0.0,0.211008,7.0
1497,nuls-io/nuls,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.496508,2.397895,0.181399,0.375446,0.150688,0.213600,0.006771,0.0,0.206189,8.0
1410,yahoo/fili,0.0,1.0,NaN,1.0,1.0,28.0,0.0,167.0,21199.0,...,0.000000,3.871201,0.069234,0.095406,0.485305,0.175566,0.215923,0.5,0.203131,8.0
1148,shifuml/shifu,0.0,0.0,0.0,0.0,1.0,19.0,0.0,205.0,113071.0,...,1.098612,3.044522,0.133631,0.203534,0.332329,0.260908,0.055134,0.0,0.201386,8.0


In [18]:
# ---------- 5. Top 200 ----------
top_200 = df_sorted.head(200)
top_200.to_csv("./top_200_projects_java.csv", index=False)

print(
    top_200[
        [
            "Name",
            "overall_score",
            "popularity_score",
            "activity_score",
            "pr_score",
            "testing_score",
            "community_score",
            "maturity_score",
            "closed_pr_ratio",
            "merged_pr_ratio",
            "test_ratio",
            "unique_pr_authors",
            "has_actions",
            "workflow_runs_tests",
            "has_contributing",
        ]
    ].head(20)
)

                                    Name  overall_score  popularity_score  \
1234    spring-projects/spring-framework       0.957954          0.996205   
1064                   quarkusio/quarkus       0.955957          0.934821   
1224         spring-projects/spring-boot       0.945562          0.998512   
268                checkstyle/checkstyle       0.928997          0.912723   
142                apache/shardingsphere       0.924042          0.960938   
735                    keycloak/keycloak       0.913662          0.978795   
367                dropwizard/dropwizard       0.913258          0.906101   
1143                 seleniumhq/selenium       0.909629          0.981771   
950       openapitools/openapi-generator       0.909445          0.970685   
79                          apache/dubbo       0.909020          0.992634   
135                        apache/pulsar       0.907525          0.940774   
29                         alibaba/nacos       0.906345          0.984524   